# Sensitivity Analysis Using Raw Full-Match Totals

This notebook evaluates whether the main conclusions are robust when exposure-dependent performance indicators are represented as raw full-match totals rather than per-90 values.

The modeling framework, grouping strategy, random seed, outcome definition, and evaluation principles follow the primary analysis.


In [ ]:
# ============================================================
# SENSITIVITY ANALYSIS — RAW FULL-MATCH TOTALS
# STEP 1: LOAD AND AUDIT CORRECTED DATA
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

RANDOM_STATE = 42

REPO_ROOT = Path.cwd().resolve().parent


RAW_DATA_PATH = (
    REPO_ROOT
    / "data"
    / "FIFA_WC2026_Analysis_Ready_Corrected.xlsx"
)

RESULTS_DIR = REPO_ROOT / "results"
FIGURES_DIR = REPO_ROOT / "figures"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

if not RAW_DATA_PATH.exists():
    raise FileNotFoundError(
        "Required raw-total source dataset was not found:\n"
        f"{RAW_DATA_PATH}\n\n"
        "Place FIFA_WC2026_Analysis_Ready_Corrected.xlsx "
        "inside the repository data/ directory. "
        "See data/README.md for details."
    )

print("SENSITIVITY ANALYSIS — RAW TOTALS")
print("=" * 75)

print("\nDataset path:")
print(RAW_DATA_PATH)

print("\nFile exists:", RAW_DATA_PATH.exists())

# Load corrected raw-total dataset
df_raw = pd.read_excel(RAW_DATA_PATH)

print("\nDataset shape:", df_raw.shape)
print("Unique matches:", df_raw["Match_ID"].nunique())


# ============================================================
# DEFINE RAW-TOTAL PREDICTORS
# ============================================================

raw_predictors = [
    "Ranking_Difference",
    "Attempts_on_Target",
    "Corner_Kicks",
    "Crosses",
    "Ball_Possession_Percent",
    "Completed_Passes",
    "Completed_Line_Breaks",
    "Defensive_Pressures",
    "Forced_Turnovers",
    "Second_Balls",
    "Saves",
    "Save_Percentage",
    "Distance_Covered_km",
    "Zone4_Low_Speed_Sprinting"
]

outcome_col = "Win_Binary"
group_col = "Match_ID"
phase_col = "Tournament_Phase"


# ============================================================
# REQUIRED-COLUMN AUDIT
# ============================================================

required_columns = (
    raw_predictors
    + [outcome_col, group_col, phase_col]
)

missing_columns = [
    col for col in required_columns
    if col not in df_raw.columns
]

print("\nRequired-column audit")
print("-" * 50)

print("Predictors:", len(raw_predictors))
print("Missing required columns:", missing_columns)


# ============================================================
# OUTCOME AUDIT
# ============================================================

print("\nOutcome distribution")
print("-" * 50)

print(
    df_raw[outcome_col]
    .value_counts()
    .sort_index()
)

print("\nOutcome by tournament phase:")
print(
    pd.crosstab(
        df_raw[phase_col],
        df_raw[outcome_col],
        margins=True
    )
)


# ============================================================
# PENALTY-SHOOTOUT CORRECTION AUDIT
# ============================================================

penalty_rows = df_raw[
    df_raw["Penalty_Shootout"]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin(["yes", "1", "true"])
]

print("\nPenalty-shootout audit")
print("-" * 50)

print(
    "Penalty-shootout observations:",
    len(penalty_rows)
)

print(
    "Penalty-shootout matches:",
    penalty_rows["Match_ID"].nunique()
)

print(
    "Penalty observations coded Win_Binary = 0:",
    (penalty_rows[outcome_col] == 0).sum(),
    "of",
    len(penalty_rows)
)


# ============================================================
# MISSING / INFINITE VALUE AUDIT
# ============================================================

print("\nPredictor missingness")
print("-" * 50)

missingness = (
    df_raw[raw_predictors]
    .isna()
    .sum()
)

print(
    missingness[
        missingness > 0
    ]
)

numeric_matrix = (
    df_raw[raw_predictors]
    .select_dtypes(include=[np.number])
)

print(
    "\nInfinite predictor values:",
    np.isinf(numeric_matrix.to_numpy()).sum()
)


# ============================================================
# MATCH-PAIR AUDIT
# ============================================================

match_sizes = (
    df_raw.groupby("Match_ID")
    .size()
)

print("\nMatch-pair audit")
print("-" * 50)

print(
    "Matches with exactly 2 observations:",
    (match_sizes == 2).sum(),
    "of",
    len(match_sizes)
)

print(
    "Malformed matches:",
    (match_sizes != 2).sum()
)


# ============================================================
# CREATE ANALYSIS OBJECTS
# ============================================================

X_raw = df_raw[raw_predictors].copy()
y_raw = df_raw[outcome_col].astype(int).copy()
groups_raw = df_raw[group_col].copy()
phase_raw = df_raw[phase_col].copy()

print("\nAnalysis objects")
print("-" * 50)

print("X_raw:", X_raw.shape)
print("y_raw:", y_raw.shape)
print("groups_raw:", groups_raw.shape)
print("phase_raw:", phase_raw.shape)

print("\nSTEP 1 AUDIT COMPLETE")


## Raw-total versus per-90 data verification

The raw and per-90 datasets are checked for row alignment, identical outcome coding, unchanged non-exposure predictors, and the expected transformation of exposure-dependent predictors.


In [ ]:
# ============================================================
# STEP 2: VERIFY RAW-TOTAL vs PER-90 REPRESENTATIONS
# ============================================================

PER90_DATA_PATH = (
    REPO_ROOT
    / "data"
    / "FIFA_WC2026_Analysis_Ready_Per90_Corrected.xlsx"
)

if not PER90_DATA_PATH.exists():
    raise FileNotFoundError(
        "Required per-90 dataset was not found:\n"
        f"{PER90_DATA_PATH}\n\n"
        "Place FIFA_WC2026_Analysis_Ready_Per90_Corrected.xlsx "
        "inside the repository data/ directory. "
        "See data/README.md for details."
    )

df_per90_check = pd.read_excel(PER90_DATA_PATH)

print("RAW-TOTAL vs PER-90 REPRESENTATION AUDIT")
print("=" * 75)

print("Per-90 file exists:", PER90_DATA_PATH.exists())
print("Raw shape:", df_raw.shape)
print("Per-90 shape:", df_per90_check.shape)


# ------------------------------------------------------------
# 1. Verify row alignment
# ------------------------------------------------------------

same_match_order = (
    df_raw["Match_ID"].astype(str).to_numpy()
    ==
    df_per90_check["Match_ID"].astype(str).to_numpy()
).all()

same_team_order = (
    df_raw["Team"].astype(str).to_numpy()
    ==
    df_per90_check["Team"].astype(str).to_numpy()
).all()

same_opponent_order = (
    df_raw["Opponent"].astype(str).to_numpy()
    ==
    df_per90_check["Opponent"].astype(str).to_numpy()
).all()

print("\nROW ALIGNMENT")
print("-" * 50)
print("Same Match_ID order:", same_match_order)
print("Same Team order:", same_team_order)
print("Same Opponent order:", same_opponent_order)


# ------------------------------------------------------------
# 2. Verify outcome is identical
# ------------------------------------------------------------

same_outcome = np.array_equal(
    df_raw["Win_Binary"].to_numpy(),
    df_per90_check["Win_Binary"].to_numpy()
)

print("\nOUTCOME")
print("-" * 50)
print("Win_Binary identical:", same_outcome)


# ------------------------------------------------------------
# 3. Variables that should remain unchanged
# ------------------------------------------------------------

unchanged_pairs = {
    "Ranking_Difference":
        ("Ranking_Difference", "Ranking_Difference"),

    "Ball_Possession":
        ("Ball_Possession_Percent", "Ball_Possession_%"),

    "Save_Percentage":
        ("Save_Percentage", "Save_percentage")
}

print("\nUNCHANGED PREDICTORS")
print("-" * 50)

for label, (raw_col, per90_col) in unchanged_pairs.items():

    raw_values = pd.to_numeric(
        df_raw[raw_col],
        errors="coerce"
    ).to_numpy()

    per90_values = pd.to_numeric(
        df_per90_check[per90_col],
        errors="coerce"
    ).to_numpy()

    identical = np.allclose(
        raw_values,
        per90_values,
        equal_nan=True
    )

    print(f"{label}: {identical}")


# ------------------------------------------------------------
# 4. Verify the 11 intended exposure-dependent mappings
# ------------------------------------------------------------

exposure_pairs = {
    "Attempts_on_Target":
        "Attempts_on_Target_per90",

    "Corner_Kicks":
        "Corner_Kicks_per90",

    "Crosses":
        "Crosses_per90",

    "Completed_Passes":
        "Completed_Passes_per90",

    "Completed_Line_Breaks":
        "Completed_Line_Breaks_per90",

    "Defensive_Pressures":
        "Defensive_Pressures_per90",

    "Forced_Turnovers":
        "Forced_Turnovers_per90",

    "Second_Balls":
        "Second_Balls_per90",

    "Saves":
        "Saves_per90",

    "Distance_Covered_km":
        "Distance_per_90_recalculated",

    "Zone4_Low_Speed_Sprinting":
        "Zone4_per90"
}

print("\nEXPOSURE-DEPENDENT PREDICTORS")
print("-" * 75)

audit_rows = []

for raw_col, per90_col in exposure_pairs.items():

    raw_values = pd.to_numeric(
        df_raw[raw_col],
        errors="coerce"
    ).to_numpy(dtype=float)

    observed_per90 = pd.to_numeric(
        df_per90_check[per90_col],
        errors="coerce"
    ).to_numpy(dtype=float)

    exposure_factor = pd.to_numeric(
        df_per90_check["Exposure_Factor"],
        errors="coerce"
    ).to_numpy(dtype=float)

    expected_per90 = raw_values * exposure_factor

    matches_expected = np.isclose(
        observed_per90,
        expected_per90,
        equal_nan=True
    )

    audit_rows.append({
        "Raw_Variable": raw_col,
        "Per90_Variable": per90_col,
        "Rows_Matching": int(matches_expected.sum()),
        "Total_Rows": len(matches_expected),
        "All_Match": bool(matches_expected.all())
    })

representation_audit = pd.DataFrame(audit_rows)

display(representation_audit)


# ------------------------------------------------------------
# 5. Exposure factor verification
# ------------------------------------------------------------

expected_exposure = (
    90 /
    pd.to_numeric(
        df_per90_check["Match_Duration"],
        errors="coerce"
    )
)

exposure_correct = np.allclose(
    expected_exposure,
    df_per90_check["Exposure_Factor"],
    equal_nan=True
)

print("\nEXPOSURE FACTOR")
print("-" * 50)

print(
    "Exposure_Factor = 90 / Match_Duration:",
    exposure_correct
)

print(
    "Exposure factors:",
    sorted(
        df_per90_check["Exposure_Factor"]
        .dropna()
        .unique()
    )
)


print("\n" + "=" * 75)
print("STEP 2 AUDIT COMPLETE")


## Data discrepancy investigation and correction

A discrepancy involving match M14 (Cabo Verde vs Spain) is audited before the sensitivity analysis.

The verified correction is:

- Completed passes: **272 → 306**
- Completed line breaks: **306 → 57**

The correction is performed explicitly in this notebook to preserve an auditable data-processing record.


In [ ]:
# ============================================================
# STEP 2B: INVESTIGATE THE TWO REPRESENTATION MISMATCHES
# ============================================================

print("INVESTIGATING PER-90 REPRESENTATION MISMATCHES")
print("=" * 85)

problem_pairs = {
    "Completed_Passes": "Completed_Passes_per90",
    "Completed_Line_Breaks": "Completed_Line_Breaks_per90"
}

mismatch_records = []

for raw_col, per90_col in problem_pairs.items():

    raw_values = pd.to_numeric(
        df_raw[raw_col],
        errors="coerce"
    ).to_numpy(dtype=float)

    observed_per90 = pd.to_numeric(
        df_per90_check[per90_col],
        errors="coerce"
    ).to_numpy(dtype=float)

    exposure_factor = pd.to_numeric(
        df_per90_check["Exposure_Factor"],
        errors="coerce"
    ).to_numpy(dtype=float)

    expected_per90 = raw_values * exposure_factor

    matches = np.isclose(
        observed_per90,
        expected_per90,
        equal_nan=True
    )

    mismatch_indices = np.where(~matches)[0]

    print(f"\nVARIABLE: {raw_col}")
    print("-" * 70)
    print("Number of mismatches:", len(mismatch_indices))

    for idx in mismatch_indices:

        difference = (
            observed_per90[idx]
            - expected_per90[idx]
        )

        record = {
            "Row_Index": idx,
            "Match_ID": df_raw.iloc[idx]["Match_ID"],
            "Team": df_raw.iloc[idx]["Team"],
            "Opponent": df_raw.iloc[idx]["Opponent"],
            "Tournament_Phase":
                df_raw.iloc[idx]["Tournament_Phase"],
            "Extra_Time":
                df_raw.iloc[idx]["Extra_Time"],
            "Raw_Variable": raw_col,
            "Raw_Value": raw_values[idx],
            "Match_Duration":
                df_per90_check.iloc[idx]["Match_Duration"],
            "Exposure_Factor":
                exposure_factor[idx],
            "Expected_Per90":
                expected_per90[idx],
            "Stored_Per90":
                observed_per90[idx],
            "Difference":
                difference
        }

        mismatch_records.append(record)

        print("Row index:", idx)
        print("Match:", record["Match_ID"])
        print(
            "Team:",
            record["Team"],
            "vs",
            record["Opponent"]
        )
        print("Phase:", record["Tournament_Phase"])
        print("Extra time:", record["Extra_Time"])
        print("Raw value:", record["Raw_Value"])
        print("Match duration:", record["Match_Duration"])
        print("Exposure factor:", record["Exposure_Factor"])
        print("Expected per90:", record["Expected_Per90"])
        print("Stored per90:", record["Stored_Per90"])
        print("Difference:", record["Difference"])


mismatch_audit = pd.DataFrame(mismatch_records)

print("\n" + "=" * 85)
print("MISMATCH SUMMARY")
print("=" * 85)

display(mismatch_audit)


In [ ]:
# ============================================================
# STEP 2C: FULL SOURCE AUDIT — M14 CABO VERDE vs SPAIN
# ============================================================

print("M14 SOURCE-VALUE AUDIT")
print("=" * 90)

# Raw corrected row
raw_m14 = df_raw[
    (df_raw["Match_ID"] == "M14") &
    (df_raw["Team"] == "Cabo Verde")
]

# Per-90 corrected row
per90_m14 = df_per90_check[
    (df_per90_check["Match_ID"] == "M14") &
    (df_per90_check["Team"] == "Cabo Verde")
]

print("\nRAW CORRECTED DATASET")
print("-" * 60)

raw_columns_to_check = [
    "Match_ID",
    "Team",
    "Opponent",
    "Completed_Passes",
    "Completed_Line_Breaks",
    "Defensive_Pressures",
    "Forced_Turnovers",
    "Second_Balls"
]

display(
    raw_m14[raw_columns_to_check].T
)


print("\nPER-90 CORRECTED DATASET")
print("-" * 60)

per90_columns_to_check = [
    "Match_ID",
    "Team",
    "Opponent",
    "Copmleted_Pass",
    "Completed_Line_Breaks",
    "Completed_Passes_per90",
    "Completed_Line_Breaks_per90",
    "Defensive_Pressures_per90",
    "Forced_Turnovers_per90",
    "Second_Balls_per90",
    "Match_Duration",
    "Exposure_Factor"
]

display(
    per90_m14[per90_columns_to_check].T
)


# ============================================================
# ALSO SHOW BOTH TEAMS FROM MATCH M14
# ============================================================

print("\nFULL M14 — BOTH TEAMS, RAW DATA")
print("-" * 60)

display(
    df_raw.loc[
        df_raw["Match_ID"] == "M14",
        [
            "Team",
            "Opponent",
            "Completed_Passes",
            "Completed_Line_Breaks",
            "Defensive_Pressures",
            "Forced_Turnovers",
            "Second_Balls"
        ]
    ]
)


print("\nFULL M14 — BOTH TEAMS, PER-90 DATA")
print("-" * 60)

display(
    df_per90_check.loc[
        df_per90_check["Match_ID"] == "M14",
        [
            "Team",
            "Opponent",
            "Copmleted_Pass",
            "Completed_Line_Breaks",
            "Completed_Passes_per90",
            "Completed_Line_Breaks_per90"
        ]
    ]
)

print("\nDO NOT MODIFY DATA YET.")


In [ ]:
# ============================================================
# STEP 2D: CORRECT M14 CABO VERDE RAW DATA
# ============================================================

from datetime import datetime

mask_m14_cv = (
    (df_raw["Match_ID"] == "M14")
    & (df_raw["Team"] == "Cabo Verde")
    & (df_raw["Opponent"] == "Spain")
)

print("Rows identified:", mask_m14_cv.sum())

# Record old values before correction
old_passes = df_raw.loc[
    mask_m14_cv, "Completed_Passes"
].iloc[0]

old_line_breaks = df_raw.loc[
    mask_m14_cv, "Completed_Line_Breaks"
].iloc[0]

# Correct verified values
df_raw.loc[
    mask_m14_cv, "Completed_Passes"
] = 306

df_raw.loc[
    mask_m14_cv, "Completed_Line_Breaks"
] = 57


# ------------------------------------------------------------
# Create correction audit
# ------------------------------------------------------------

m14_correction_audit = pd.DataFrame([
    {
        "Match_ID": "M14",
        "Team": "Cabo Verde",
        "Opponent": "Spain",
        "Variable": "Completed_Passes",
        "Old_Value": old_passes,
        "Corrected_Value": 306,
        "Reason": "Verified source value"
    },
    {
        "Match_ID": "M14",
        "Team": "Cabo Verde",
        "Opponent": "Spain",
        "Variable": "Completed_Line_Breaks",
        "Old_Value": old_line_breaks,
        "Corrected_Value": 57,
        "Reason": "Verified source value"
    }
])


# ------------------------------------------------------------
# Save NEW corrected raw dataset
# Do not overwrite earlier file
# ------------------------------------------------------------

FINAL_RAW_PATH = (
    RESULTS_DIR
    / "FIFA_WC2026_Analysis_Ready_Raw_Final.xlsx"
)

df_raw.to_excel(
    FINAL_RAW_PATH,
    index=False
)

AUDIT_PATH = (
    RESULTS_DIR
    / "M14_CaboVerde_Data_Correction_Audit.xlsx"
)

m14_correction_audit.to_excel(
    AUDIT_PATH,
    index=False
)


# ------------------------------------------------------------
# Verify against per-90 source
# ------------------------------------------------------------

raw_m14_corrected = df_raw.loc[
    mask_m14_cv,
    ["Completed_Passes", "Completed_Line_Breaks"]
].iloc[0]

per90_m14_verified = df_per90_check.loc[
    (df_per90_check["Match_ID"] == "M14")
    & (df_per90_check["Team"] == "Cabo Verde"),
    ["Completed_Passes_per90",
     "Completed_Line_Breaks_per90"]
].iloc[0]

print("\nCORRECTION AUDIT")
print("=" * 70)

display(m14_correction_audit)

print("\nCorrected raw values:")
print(raw_m14_corrected)

print("\nExisting per-90 values:")
print(per90_m14_verified)

print("\nFILES SAVED")
print("Final raw dataset:", FINAL_RAW_PATH.exists())
print("Correction audit:", AUDIT_PATH.exists())


In [ ]:
# ============================================================
# STEP 2E: FINAL RAW vs PER-90 VERIFICATION AFTER M14 CORRECTION
# ============================================================

print("FINAL RAW vs PER-90 VERIFICATION")
print("=" * 80)

final_audit_rows = []

for raw_col, per90_col in exposure_pairs.items():

    raw_values = pd.to_numeric(
        df_raw[raw_col],
        errors="coerce"
    ).to_numpy(dtype=float)

    per90_values = pd.to_numeric(
        df_per90_check[per90_col],
        errors="coerce"
    ).to_numpy(dtype=float)

    exposure = pd.to_numeric(
        df_per90_check["Exposure_Factor"],
        errors="coerce"
    ).to_numpy(dtype=float)

    expected_per90 = raw_values * exposure

    matches = np.isclose(
        per90_values,
        expected_per90,
        equal_nan=True
    )

    final_audit_rows.append({
        "Raw_Variable": raw_col,
        "Per90_Variable": per90_col,
        "Rows_Matching": int(matches.sum()),
        "Total_Rows": len(matches),
        "All_Match": bool(matches.all())
    })

final_representation_audit = pd.DataFrame(
    final_audit_rows
)

display(final_representation_audit)

print("\n" + "=" * 80)

all_correct = (
    final_representation_audit["All_Match"].all()
)

print(
    "ALL 11 EXPOSURE-DEPENDENT VARIABLES CORRECT:",
    all_correct
)

print(
    "Outcome identical:",
    np.array_equal(
        df_raw["Win_Binary"].to_numpy(),
        df_per90_check["Win_Binary"].to_numpy()
    )
)

print(
    "Match order identical:",
    np.array_equal(
        df_raw["Match_ID"].to_numpy(),
        df_per90_check["Match_ID"].to_numpy()
    )
)

print(
    "Team order identical:",
    np.array_equal(
        df_raw["Team"].to_numpy(),
        df_per90_check["Team"].to_numpy()
    )
)

print("\nFinal raw dataset:")
print(FINAL_RAW_PATH)

if all_correct:
    print(
        "\nPASS: Raw-total dataset is ready "
        "for sensitivity analysis."
    )
else:
    print(
        "\nSTOP: At least one representation "
        "still requires investigation."
    )


## Nested match-grouped machine-learning analysis

The raw-total sensitivity analysis uses the same nested stratified match-grouped cross-validation framework as the primary analysis.


In [ ]:
# ============================================================
# STEP 3: SET UP NESTED MATCH-GROUPED CV + MODEL GRIDS
# RAW-TOTAL SENSITIVITY ANALYSIS
# ============================================================

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# ------------------------------------------------------------
# 1. CV DESIGN
# ------------------------------------------------------------

outer_cv_raw = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

inner_cv_raw = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)


# ------------------------------------------------------------
# 2. ELASTIC NET
# Median imputation + standardization within training folds
# ------------------------------------------------------------

elastic_net_raw = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        solver="saga",
        penalty="elasticnet",
        max_iter=10000,
        random_state=RANDOM_STATE
    ))
])

elastic_net_grid_raw = {
    "model__C": [
        0.01, 0.1, 1, 10, 100
    ],
    "model__l1_ratio": [
        0, 0.25, 0.50, 0.75, 1.0
    ]
}


# ------------------------------------------------------------
# 3. RANDOM FOREST
# Median imputation only
# ------------------------------------------------------------

random_forest_raw = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

random_forest_grid_raw = {
    "model__n_estimators": [300, 500],
    "model__max_depth": [None, 3, 5, 8],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": ["sqrt", 0.5, 1.0]
}


# ------------------------------------------------------------
# 4. XGBOOST
# Median imputation only
# ------------------------------------------------------------

xgboost_raw = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", XGBClassifier(
        random_state=RANDOM_STATE,
        eval_metric="logloss",
        n_jobs=-1
    ))
])

xgboost_grid_raw = {
    "model__n_estimators": [100, 200, 300],
    "model__learning_rate": [0.01, 0.05, 0.10],
    "model__max_depth": [2, 3, 4],
    "model__min_child_weight": [1, 3, 5],
    "model__subsample": [0.70, 1.0],
    "model__colsample_bytree": [0.70, 1.0]
}


# ------------------------------------------------------------
# 5. MODEL DICTIONARY
# ------------------------------------------------------------

models_raw = {
    "Elastic Net": (
        elastic_net_raw,
        elastic_net_grid_raw
    ),
    "Random Forest": (
        random_forest_raw,
        random_forest_grid_raw
    ),
    "XGBoost": (
        xgboost_raw,
        xgboost_grid_raw
    )
}


# ------------------------------------------------------------
# 6. GRID-SIZE AUDIT
# ------------------------------------------------------------

from math import prod

def grid_size(grid):
    return prod(
        len(values)
        for values in grid.values()
    )

print("RAW-TOTAL SENSITIVITY MODEL SETUP")
print("=" * 75)

for model_name, (_, grid) in models_raw.items():
    print(
        f"{model_name}: "
        f"{grid_size(grid)} parameter combinations"
    )


# ------------------------------------------------------------
# 7. OUTER-FOLD GROUPING AUDIT
# ------------------------------------------------------------

raw_fold_audit = []

for fold, (train_idx, test_idx) in enumerate(
    outer_cv_raw.split(
        X_raw,
        y_raw,
        groups_raw
    ),
    start=1
):

    train_groups = set(
        groups_raw.iloc[train_idx]
    )

    test_groups = set(
        groups_raw.iloc[test_idx]
    )

    overlap = train_groups.intersection(
        test_groups
    )

    raw_fold_audit.append({
        "Fold": fold,
        "Train_Observations": len(train_idx),
        "Test_Observations": len(test_idx),
        "Train_Matches": len(train_groups),
        "Test_Matches": len(test_groups),
        "Train_Wins": int(
            y_raw.iloc[train_idx].sum()
        ),
        "Train_NonWins": int(
            len(train_idx)
            - y_raw.iloc[train_idx].sum()
        ),
        "Test_Wins": int(
            y_raw.iloc[test_idx].sum()
        ),
        "Test_NonWins": int(
            len(test_idx)
            - y_raw.iloc[test_idx].sum()
        ),
        "Match_Overlap": len(overlap)
    })

raw_fold_audit = pd.DataFrame(
    raw_fold_audit
)

print("\nOUTER-FOLD AUDIT")
print("=" * 75)

display(raw_fold_audit)

print(
    "\nTotal test observations:",
    raw_fold_audit[
        "Test_Observations"
    ].sum()
)

print(
    "All folds have zero match overlap:",
    bool(
        (raw_fold_audit[
            "Match_Overlap"
        ] == 0).all()
    )
)


In [ ]:
# ============================================================
# STEP 4: FULL NESTED CV — RAW-TOTAL SENSITIVITY ANALYSIS
# ============================================================

from sklearn.model_selection import GridSearchCV
from sklearn.metrics import roc_auc_score
import numpy as np
import pandas as pd
import time

print("RAW-TOTAL SENSITIVITY — FULL NESTED CV")
print("=" * 80)

# ------------------------------------------------------------
# STORAGE
# ------------------------------------------------------------

raw_oof_predictions = {
    model_name: np.full(len(X_raw), np.nan)
    for model_name in models_raw
}

raw_fold_results = []
raw_best_parameters = []

# IMPORTANT: retain fitted outer-fold models and test indices
raw_fitted_outer_models = {
    model_name: []
    for model_name in models_raw
}

raw_outer_test_indices = {
    model_name: []
    for model_name in models_raw
}


# ------------------------------------------------------------
# MODEL LOOP
# ------------------------------------------------------------

for model_name, (pipeline, param_grid) in models_raw.items():

    model_start = time.time()

    print("\n" + "=" * 80)
    print("MODEL:", model_name)
    print("=" * 80)

    for outer_fold, (train_idx, test_idx) in enumerate(
        outer_cv_raw.split(
            X_raw,
            y_raw,
            groups_raw
        ),
        start=1
    ):

        fold_start = time.time()

        X_outer_train = X_raw.iloc[train_idx]
        X_outer_test = X_raw.iloc[test_idx]

        y_outer_train = y_raw.iloc[train_idx]
        y_outer_test = y_raw.iloc[test_idx]

        groups_outer_train = groups_raw.iloc[train_idx]

        # ----------------------------------------------------
        # Inner grouped hyperparameter search
        # ----------------------------------------------------

        grid_search = GridSearchCV(
            estimator=pipeline,
            param_grid=param_grid,
            scoring="roc_auc",
            cv=inner_cv_raw,
            n_jobs=-1,
            refit=True,
            return_train_score=False
        )

        grid_search.fit(
            X_outer_train,
            y_outer_train,
            groups=groups_outer_train
        )

        # Best fitted pipeline for this outer fold
        best_model = grid_search.best_estimator_

        # ----------------------------------------------------
        # Held-out outer-fold prediction
        # ----------------------------------------------------

        fold_probability = best_model.predict_proba(
            X_outer_test
        )[:, 1]

        raw_oof_predictions[
            model_name
        ][test_idx] = fold_probability

        fold_auc = roc_auc_score(
            y_outer_test,
            fold_probability
        )

        # ----------------------------------------------------
        # Store fold results
        # ----------------------------------------------------

        raw_fold_results.append({
            "Model": model_name,
            "Outer_Fold": outer_fold,
            "Test_Observations": len(test_idx),
            "ROC_AUC": fold_auc,
            "Best_Inner_ROC_AUC":
                grid_search.best_score_
        })

        # Store best parameters
        parameter_record = {
            "Model": model_name,
            "Outer_Fold": outer_fold
        }

        parameter_record.update(
            grid_search.best_params_
        )

        raw_best_parameters.append(
            parameter_record
        )

        # ----------------------------------------------------
        # IMPORTANT FOR LATER OOF SHAP
        # ----------------------------------------------------

        raw_fitted_outer_models[
            model_name
        ].append(best_model)

        raw_outer_test_indices[
            model_name
        ].append(np.array(test_idx))

        fold_elapsed = (
            time.time() - fold_start
        )

        print(
            f"Fold {outer_fold}/5 | "
            f"Test AUC = {fold_auc:.3f} | "
            f"Best inner AUC = "
            f"{grid_search.best_score_:.3f} | "
            f"Time = {fold_elapsed / 60:.2f} min"
        )


    model_elapsed = (
        time.time() - model_start
    )

    print(
        f"\n{model_name} completed in "
        f"{model_elapsed / 60:.2f} minutes."
    )


# ============================================================
# COMPLETENESS AUDIT
# ============================================================

print("\n" + "=" * 80)
print("RAW-TOTAL NESTED CV COMPLETENESS AUDIT")
print("=" * 80)

for model_name in models_raw:

    predictions = raw_oof_predictions[
        model_name
    ]

    print(
        f"{model_name}: "
        f"{np.isfinite(predictions).sum()}/"
        f"{len(predictions)} OOF predictions"
    )

    print(
        f"  Stored fitted outer models: "
        f"{len(raw_fitted_outer_models[model_name])}"
    )

    print(
        f"  Stored test-index sets: "
        f"{len(raw_outer_test_indices[model_name])}"
    )


raw_fold_results_df = pd.DataFrame(
    raw_fold_results
)

raw_best_parameters_df = pd.DataFrame(
    raw_best_parameters
)

print("\nFOLD RESULTS")
display(raw_fold_results_df.round(4))

print("\nBEST PARAMETERS")
display(raw_best_parameters_df)

print("\nSTEP 4 COMPLETE")


## Out-of-fold model performance

Performance is evaluated from pooled held-out predictions, followed by tournament-phase evaluation and match-level bootstrap confidence intervals.


In [ ]:
# ============================================================
# STEP 5: POOLED OOF PERFORMANCE — RAW-TOTAL SENSITIVITY
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    brier_score_loss
)

# ------------------------------------------------------------
# Expected Calibration Error
# ------------------------------------------------------------

def calculate_ece(y_true, y_prob, n_bins=10):

    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)

    bins = np.linspace(0, 1, n_bins + 1)

    bin_ids = np.digitize(
        y_prob,
        bins[1:-1],
        right=False
    )

    ece = 0.0

    for b in range(n_bins):

        mask = bin_ids == b

        if mask.sum() == 0:
            continue

        observed = y_true[mask].mean()
        predicted = y_prob[mask].mean()

        ece += (
            mask.sum() / len(y_true)
        ) * abs(observed - predicted)

    return ece


# ------------------------------------------------------------
# Calculate pooled OOF metrics
# ------------------------------------------------------------

raw_performance_rows = []

for model_name, probabilities in raw_oof_predictions.items():

    probabilities = np.asarray(probabilities)

    predicted_class = (
        probabilities >= 0.50
    ).astype(int)

    raw_performance_rows.append({
        "Model": model_name,
        "N": len(y_raw),
        "Accuracy": accuracy_score(
            y_raw,
            predicted_class
        ),
        "Precision": precision_score(
            y_raw,
            predicted_class,
            zero_division=0
        ),
        "Recall": recall_score(
            y_raw,
            predicted_class,
            zero_division=0
        ),
        "F1": f1_score(
            y_raw,
            predicted_class,
            zero_division=0
        ),
        "ROC_AUC": roc_auc_score(
            y_raw,
            probabilities
        ),
        "Brier": brier_score_loss(
            y_raw,
            probabilities
        ),
        "ECE": calculate_ece(
            y_raw,
            probabilities,
            n_bins=10
        )
    })


raw_performance_df = pd.DataFrame(
    raw_performance_rows
)


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print(
    "RAW-TOTAL SENSITIVITY — "
    "POOLED OOF PERFORMANCE"
)

print("=" * 80)

display(
    raw_performance_df.round(3)
)


# ------------------------------------------------------------
# Basic audit
# ------------------------------------------------------------

print("\nAUDIT")
print("=" * 80)

for model_name in models_raw:

    p = np.asarray(
        raw_oof_predictions[model_name]
    )

    print(
        f"{model_name}: "
        f"N={len(p)}, "
        f"missing={np.isnan(p).sum()}, "
        f"min={p.min():.4f}, "
        f"max={p.max():.4f}"
    )


In [ ]:
# ============================================================
# STEP 6: PHASE-SPECIFIC OOF PERFORMANCE
# RAW-TOTAL SENSITIVITY ANALYSIS
# ============================================================

print(
    "RAW-TOTAL SENSITIVITY — "
    "PHASE-SPECIFIC OOF PERFORMANCE"
)
print("=" * 85)

# Check phase labels first
print("\nPhase distribution:")
print(df_raw["Tournament_Phase"].value_counts())

print("\nPhase × outcome:")
print(
    pd.crosstab(
        df_raw["Tournament_Phase"],
        y_raw
    )
)


# ------------------------------------------------------------
# Calculate metrics within each tournament phase
# ------------------------------------------------------------

raw_phase_rows = []

for phase_name in df_raw[
    "Tournament_Phase"
].dropna().unique():

    phase_mask = (
        df_raw["Tournament_Phase"] == phase_name
    ).to_numpy()

    y_phase = np.asarray(
        y_raw
    )[phase_mask]

    for model_name, probabilities in (
        raw_oof_predictions.items()
    ):

        probabilities = np.asarray(
            probabilities
        )

        phase_prob = probabilities[
            phase_mask
        ]

        phase_pred = (
            phase_prob >= 0.50
        ).astype(int)

        raw_phase_rows.append({
            "Phase": phase_name,
            "Model": model_name,
            "N": len(y_phase),
            "Wins": int(
                y_phase.sum()
            ),
            "Non_Wins": int(
                len(y_phase) - y_phase.sum()
            ),
            "Accuracy": accuracy_score(
                y_phase,
                phase_pred
            ),
            "Precision": precision_score(
                y_phase,
                phase_pred,
                zero_division=0
            ),
            "Recall": recall_score(
                y_phase,
                phase_pred,
                zero_division=0
            ),
            "F1": f1_score(
                y_phase,
                phase_pred,
                zero_division=0
            ),
            "ROC_AUC": roc_auc_score(
                y_phase,
                phase_prob
            ),
            "Brier": brier_score_loss(
                y_phase,
                phase_prob
            ),
            "ECE": calculate_ece(
                y_phase,
                phase_prob,
                n_bins=10
            )
        })


raw_phase_performance_df = pd.DataFrame(
    raw_phase_rows
)


# ------------------------------------------------------------
# Arrange Group first, Knockout second
# ------------------------------------------------------------

phase_order = [
    "Group Stage",
    "Knockout Stage"
]

raw_phase_performance_df["Phase"] = pd.Categorical(
    raw_phase_performance_df["Phase"],
    categories=phase_order,
    ordered=True
)

raw_phase_performance_df = (
    raw_phase_performance_df
    .sort_values(
        ["Phase", "Model"]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("\nPHASE-SPECIFIC PERFORMANCE")
print("=" * 85)

display(
    raw_phase_performance_df.round(3)
)


# ------------------------------------------------------------
# Audit
# ------------------------------------------------------------

print("\nAUDIT")
print("=" * 85)

for phase_name in phase_order:

    temp = raw_phase_performance_df[
        raw_phase_performance_df[
            "Phase"
        ] == phase_name
    ]

    if len(temp) > 0:

        print(
            f"{phase_name}: "
            f"N={int(temp.iloc[0]['N'])}, "
            f"Wins={int(temp.iloc[0]['Wins'])}, "
            f"Non-wins={int(temp.iloc[0]['Non_Wins'])}"
        )


In [ ]:
# ============================================================
# STEP 7: 5,000 MATCH-LEVEL BOOTSTRAP CONFIDENCE INTERVALS
# RAW-TOTAL SENSITIVITY ANALYSIS
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    brier_score_loss
)

import numpy as np
import pandas as pd

N_BOOT = 5000
BOOT_SEED = 42

rng = np.random.default_rng(BOOT_SEED)

y_array = np.asarray(y_raw)
match_array = np.asarray(groups_raw)

unique_matches = np.unique(match_array)

print("RAW-TOTAL MATCH-LEVEL BOOTSTRAP")
print("=" * 80)
print("Bootstrap repetitions:", N_BOOT)
print("Unique matches:", len(unique_matches))
print("Observations:", len(y_array))


# ------------------------------------------------------------
# Metric function
# ------------------------------------------------------------

def bootstrap_metrics(y_true, probabilities):

    predicted = (
        probabilities >= 0.50
    ).astype(int)

    return {
        "Accuracy": accuracy_score(
            y_true, predicted
        ),

        "Precision": precision_score(
            y_true,
            predicted,
            zero_division=0
        ),

        "Recall": recall_score(
            y_true,
            predicted,
            zero_division=0
        ),

        "F1": f1_score(
            y_true,
            predicted,
            zero_division=0
        ),

        "ROC_AUC": roc_auc_score(
            y_true,
            probabilities
        ),

        "Brier": brier_score_loss(
            y_true,
            probabilities
        ),

        "ECE": calculate_ece(
            y_true,
            probabilities,
            n_bins=10
        )
    }


# ------------------------------------------------------------
# Storage
# ------------------------------------------------------------

raw_bootstrap_results = {
    model_name: []
    for model_name in raw_oof_predictions
}


# ------------------------------------------------------------
# Bootstrap
# ------------------------------------------------------------

for b in range(N_BOOT):

    sampled_matches = rng.choice(
        unique_matches,
        size=len(unique_matches),
        replace=True
    )

    # Important:
    # preserve BOTH team observations for each sampled match,
    # including repeated matches in bootstrap samples.

    sampled_indices = np.concatenate([
        np.where(
            match_array == match_id
        )[0]
        for match_id in sampled_matches
    ])

    y_boot = y_array[
        sampled_indices
    ]

    # ROC-AUC requires both outcome classes
    if len(np.unique(y_boot)) < 2:
        continue

    for model_name, probabilities in (
        raw_oof_predictions.items()
    ):

        prob_array = np.asarray(
            probabilities
        )

        prob_boot = prob_array[
            sampled_indices
        ]

        metrics = bootstrap_metrics(
            y_boot,
            prob_boot
        )

        raw_bootstrap_results[
            model_name
        ].append(metrics)


# ------------------------------------------------------------
# Summarize 95% percentile CIs
# ------------------------------------------------------------

raw_bootstrap_ci_rows = []

metric_names = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "ROC_AUC",
    "Brier",
    "ECE"
]

for model_name in raw_oof_predictions:

    boot_df = pd.DataFrame(
        raw_bootstrap_results[
            model_name
        ]
    )

    point_row = (
        raw_performance_df
        .loc[
            raw_performance_df[
                "Model"
            ] == model_name
        ]
        .iloc[0]
    )

    for metric in metric_names:

        values = boot_df[
            metric
        ].dropna().to_numpy()

        raw_bootstrap_ci_rows.append({
            "Model": model_name,
            "Metric": metric,
            "Point_Estimate":
                point_row[metric],
            "CI_Lower":
                np.percentile(
                    values,
                    2.5
                ),
            "CI_Upper":
                np.percentile(
                    values,
                    97.5
                ),
            "Valid_Bootstraps":
                len(values)
        })


raw_bootstrap_ci_df = pd.DataFrame(
    raw_bootstrap_ci_rows
)


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("\nBOOTSTRAP 95% CONFIDENCE INTERVALS")
print("=" * 80)

display(
    raw_bootstrap_ci_df.round(3)
)


# ------------------------------------------------------------
# Audit
# ------------------------------------------------------------

print("\nBOOTSTRAP AUDIT")
print("=" * 80)

for model_name in raw_oof_predictions:

    print(
        model_name,
        ":",
        len(
            raw_bootstrap_results[
                model_name
            ]
        ),
        "valid bootstrap samples"
    )


# ------------------------------------------------------------
# Save now
# ------------------------------------------------------------

RAW_BOOTSTRAP_PATH = (
    RESULTS_DIR
    / "Sensitivity_Raw_Model_Performance_Bootstrap_CI.xlsx"
)

raw_bootstrap_ci_df.to_excel(
    RAW_BOOTSTRAP_PATH,
    index=False
)

print("\nSaved:")
print(RAW_BOOTSTRAP_PATH)


In [ ]:
# ============================================================
# STEP 8: PAIRED MATCH-LEVEL BOOTSTRAP MODEL COMPARISONS
# RAW-TOTAL SENSITIVITY ANALYSIS
# ============================================================

from itertools import combinations

N_BOOT_PAIRED = 5000
PAIR_SEED = 42

rng_pair = np.random.default_rng(PAIR_SEED)

model_pairs = list(
    combinations(
        raw_oof_predictions.keys(),
        2
    )
)

metrics_for_comparison = [
    "ROC_AUC",
    "Accuracy",
    "F1",
    "Brier"
]

paired_differences = {
    (m1, m2): {
        metric: []
        for metric in metrics_for_comparison
    }
    for m1, m2 in model_pairs
}


# ------------------------------------------------------------
# Bootstrap paired differences
# ------------------------------------------------------------

for b in range(N_BOOT_PAIRED):

    sampled_matches = rng_pair.choice(
        unique_matches,
        size=len(unique_matches),
        replace=True
    )

    sampled_indices = np.concatenate([
        np.where(
            match_array == match_id
        )[0]
        for match_id in sampled_matches
    ])

    y_boot = y_array[sampled_indices]

    if len(np.unique(y_boot)) < 2:
        continue

    bootstrap_model_metrics = {}

    for model_name, probabilities in (
        raw_oof_predictions.items()
    ):

        prob_boot = np.asarray(
            probabilities
        )[sampled_indices]

        bootstrap_model_metrics[
            model_name
        ] = bootstrap_metrics(
            y_boot,
            prob_boot
        )

    # Same bootstrap sample for both models
    for model_1, model_2 in model_pairs:

        for metric in metrics_for_comparison:

            difference = (
                bootstrap_model_metrics[
                    model_1
                ][metric]
                -
                bootstrap_model_metrics[
                    model_2
                ][metric]
            )

            paired_differences[
                (model_1, model_2)
            ][metric].append(
                difference
            )


# ------------------------------------------------------------
# Point estimates + 95% paired bootstrap CI
# ------------------------------------------------------------

paired_rows = []

for model_1, model_2 in model_pairs:

    row_1 = (
        raw_performance_df
        .loc[
            raw_performance_df["Model"]
            == model_1
        ]
        .iloc[0]
    )

    row_2 = (
        raw_performance_df
        .loc[
            raw_performance_df["Model"]
            == model_2
        ]
        .iloc[0]
    )

    for metric in metrics_for_comparison:

        values = np.asarray(
            paired_differences[
                (model_1, model_2)
            ][metric]
        )

        point_difference = (
            row_1[metric]
            - row_2[metric]
        )

        ci_lower = np.percentile(
            values,
            2.5
        )

        ci_upper = np.percentile(
            values,
            97.5
        )

        paired_rows.append({
            "Model_1": model_1,
            "Model_2": model_2,
            "Metric": metric,
            "Difference_M1_minus_M2":
                point_difference,
            "CI_Lower": ci_lower,
            "CI_Upper": ci_upper,
            "CI_Excludes_Zero":
                bool(
                    (ci_lower > 0)
                    or
                    (ci_upper < 0)
                ),
            "Valid_Bootstraps":
                len(values)
        })


raw_paired_comparison_df = pd.DataFrame(
    paired_rows
)


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print(
    "RAW-TOTAL SENSITIVITY — "
    "PAIRED MODEL COMPARISONS"
)
print("=" * 90)

display(
    raw_paired_comparison_df.round(3)
)


print("\nINTERPRETATION OF DIFFERENCE")
print("=" * 90)

print(
    "Difference = Model 1 - Model 2"
)

print(
    "For Accuracy, F1 and ROC-AUC: "
    "positive favors Model 1."
)

print(
    "For Brier: negative favors Model 1 "
    "(lower Brier is better)."
)


# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

RAW_PAIRED_PATH = (
    RESULTS_DIR
    / "Sensitivity_Raw_Paired_Model_Comparisons.xlsx"
)

raw_paired_comparison_df.to_excel(
    RAW_PAIRED_PATH,
    index=False
)

print("\nSaved:")
print(RAW_PAIRED_PATH)


## Out-of-fold XGBoost SHAP analysis

SHAP values are computed for held-out observations to assess predictor importance and direction under the raw-total representation.

Positive SHAP values shift the model output toward win; negative values shift it toward non-win.


In [ ]:
# ============================================================
# STEP 9: OOF SHAP — RAW-TOTAL XGBOOST SENSITIVITY
# ============================================================

import shap
import numpy as np
import pandas as pd

print("RAW-TOTAL XGBOOST — OOF SHAP")
print("=" * 80)


# ------------------------------------------------------------
# Storage
# ------------------------------------------------------------

raw_oof_shap_values = np.full(
    (len(X_raw), X_raw.shape[1]),
    np.nan
)

raw_oof_shap_feature_values = np.full(
    (len(X_raw), X_raw.shape[1]),
    np.nan
)

raw_shap_expected_values = []


# ------------------------------------------------------------
# Retrieve saved XGBoost outer-fold models
# ------------------------------------------------------------

xgb_models_raw = raw_fitted_outer_models[
    "XGBoost"
]

xgb_test_indices_raw = raw_outer_test_indices[
    "XGBoost"
]

print(
    "Stored XGBoost models:",
    len(xgb_models_raw)
)

print(
    "Stored test-index sets:",
    len(xgb_test_indices_raw)
)


# ------------------------------------------------------------
# OOF SHAP fold by fold
# ------------------------------------------------------------

for fold_number, (
    fitted_pipeline,
    test_idx
) in enumerate(
    zip(
        xgb_models_raw,
        xgb_test_indices_raw
    ),
    start=1
):

    # Components of fitted pipeline
    fitted_imputer = fitted_pipeline.named_steps[
        "imputer"
    ]

    fitted_xgb = fitted_pipeline.named_steps[
        "model"
    ]

    # Held-out observations only
    X_test_fold = X_raw.iloc[
        test_idx
    ]

    # Apply TRAINING-FOLD fitted imputer
    X_test_imputed = fitted_imputer.transform(
        X_test_fold
    )

    # Tree SHAP
    explainer = shap.TreeExplainer(
        fitted_xgb
    )

    shap_values_fold = explainer.shap_values(
        X_test_imputed
    )

    # Ensure array
    shap_values_fold = np.asarray(
        shap_values_fold
    )

    # Store in original row positions
    raw_oof_shap_values[
        test_idx, :
    ] = shap_values_fold

    raw_oof_shap_feature_values[
        test_idx, :
    ] = X_test_imputed

    # Expected value
    expected = explainer.expected_value

    if isinstance(
        expected,
        (list, np.ndarray)
    ):
        expected = np.asarray(
            expected
        ).ravel()[0]

    raw_shap_expected_values.append(
        float(expected)
    )

    print(
        f"Fold {fold_number}: "
        f"N={len(test_idx)}, "
        f"SHAP shape={shap_values_fold.shape}, "
        f"expected={float(expected):.6f}"
    )


# ------------------------------------------------------------
# Completeness audit
# ------------------------------------------------------------

print("\nOOF SHAP COMPLETENESS AUDIT")
print("=" * 80)

print(
    "SHAP matrix shape:",
    raw_oof_shap_values.shape
)

print(
    "Feature-value matrix shape:",
    raw_oof_shap_feature_values.shape
)

complete_rows = np.all(
    np.isfinite(
        raw_oof_shap_values
    ),
    axis=1
)

print(
    "Complete SHAP rows:",
    complete_rows.sum(),
    "/",
    len(complete_rows)
)

print(
    "Missing SHAP values:",
    np.isnan(
        raw_oof_shap_values
    ).sum()
)

print(
    "Infinite SHAP values:",
    np.isinf(
        raw_oof_shap_values
    ).sum()
)


# ------------------------------------------------------------
# Check each observation appears in exactly one test fold
# ------------------------------------------------------------

all_test_indices = np.concatenate(
    xgb_test_indices_raw
)

unique_idx, counts = np.unique(
    all_test_indices,
    return_counts=True
)

print(
    "Unique held-out observations:",
    len(unique_idx)
)

print(
    "Every observation used exactly once:",
    bool(
        len(unique_idx) == len(X_raw)
        and np.all(counts == 1)
    )
)


# ------------------------------------------------------------
# Create DataFrames
# ------------------------------------------------------------

raw_oof_shap_df = pd.DataFrame(
    raw_oof_shap_values,
    columns=X_raw.columns,
    index=X_raw.index
)

raw_oof_shap_feature_df = pd.DataFrame(
    raw_oof_shap_feature_values,
    columns=X_raw.columns,
    index=X_raw.index
)


print("\nSTEP 9 COMPLETE")


In [ ]:
# ============================================================
# STEP 10: SHAP ADDITIVITY + DIRECTION AUDIT
# RAW-TOTAL XGBOOST
# ============================================================

from scipy.special import expit
import numpy as np
import pandas as pd

print("RAW-TOTAL XGBOOST — SHAP ADDITIVITY AUDIT")
print("=" * 80)

additivity_rows = []

# Store reconstructed probabilities
raw_shap_reconstructed_prob = np.full(
    len(X_raw),
    np.nan
)

xgb_oof_prob_raw = np.asarray(
    raw_oof_predictions["XGBoost"]
)


# ------------------------------------------------------------
# Fold-by-fold reconstruction
# ------------------------------------------------------------

for fold_number, test_idx in enumerate(
    xgb_test_indices_raw,
    start=1
):

    expected_value = raw_shap_expected_values[
        fold_number - 1
    ]

    # SHAP values sum on raw model-output scale
    reconstructed_raw_output = (
        expected_value
        +
        raw_oof_shap_values[
            test_idx, :
        ].sum(axis=1)
    )

    # Convert raw/log-odds output to probability
    reconstructed_probability = expit(
        reconstructed_raw_output
    )

    raw_shap_reconstructed_prob[
        test_idx
    ] = reconstructed_probability

    original_probability = (
        xgb_oof_prob_raw[test_idx]
    )

    absolute_difference = np.abs(
        reconstructed_probability
        - original_probability
    )

    additivity_rows.append({
        "Fold": fold_number,
        "N": len(test_idx),
        "Expected_Value":
            expected_value,
        "Mean_Absolute_Probability_Difference":
            absolute_difference.mean(),
        "Max_Absolute_Probability_Difference":
            absolute_difference.max()
    })


raw_shap_additivity_df = pd.DataFrame(
    additivity_rows
)


print("\nFOLD-LEVEL AUDIT")
display(
    raw_shap_additivity_df.round(6)
)


# ------------------------------------------------------------
# Overall audit
# ------------------------------------------------------------

overall_difference = np.abs(
    raw_shap_reconstructed_prob
    - xgb_oof_prob_raw
)

print("\nOVERALL AUDIT")
print("=" * 80)

print(
    "Mean absolute probability difference:",
    round(
        overall_difference.mean(),
        6
    )
)

print(
    "Maximum absolute probability difference:",
    round(
        overall_difference.max(),
        6
    )
)

print(
    "Missing reconstructed probabilities:",
    np.isnan(
        raw_shap_reconstructed_prob
    ).sum()
)


# ------------------------------------------------------------
# SHAP sign interpretation
# ------------------------------------------------------------

print("\nSHAP DIRECTION")
print("=" * 80)

print(
    "Positive SHAP value -> pushes model output "
    "toward Win_Binary = 1 (WIN)"
)

print(
    "Negative SHAP value -> pushes model output "
    "toward Win_Binary = 0 (NON-WIN)"
)

print(
    "Feature color in a beeswarm represents the "
    "FEATURE VALUE, not whether the team won."
)


# ------------------------------------------------------------
# Ranking-difference formula audit again
# ------------------------------------------------------------

ranking_expected = (
    df_raw["Opponent_FIFA_Ranking"]
    - df_raw["FIFA_Ranking"]
)

ranking_difference_error = np.abs(
    ranking_expected
    - df_raw["Ranking_Difference"]
)

print("\nRANKING DIFFERENCE AUDIT")
print("=" * 80)

print(
    "Formula:",
    "Opponent FIFA ranking - focal team FIFA ranking"
)

print(
    "Rows satisfying formula:",
    int(
        np.isclose(
            ranking_expected,
            df_raw["Ranking_Difference"]
        ).sum()
    ),
    "/",
    len(df_raw)
)

print(
    "Maximum discrepancy:",
    ranking_difference_error.max()
)

print(
    "\nInterpretation:"
)

print(
    "Positive Ranking_Difference = "
    "focal team is better ranked."
)

print(
    "Negative Ranking_Difference = "
    "focal team is worse ranked."
)


In [ ]:
# ============================================================
# STEP 11: RAW-TOTAL OOF SHAP IMPORTANCE
# OVERALL + GROUP STAGE + KNOCKOUT STAGE
# ============================================================

import numpy as np
import pandas as pd

print("RAW-TOTAL XGBOOST — OOF SHAP IMPORTANCE")
print("=" * 85)

feature_names_raw = list(X_raw.columns)

phase_array = (
    df_raw["Tournament_Phase"]
    .astype(str)
    .to_numpy()
)

group_mask = phase_array == "Group Stage"
knockout_mask = phase_array == "Knockout Stage"


# ------------------------------------------------------------
# 1. Mean absolute SHAP
# ------------------------------------------------------------

overall_mean_abs = np.mean(
    np.abs(raw_oof_shap_values),
    axis=0
)

group_mean_abs = np.mean(
    np.abs(
        raw_oof_shap_values[
            group_mask, :
        ]
    ),
    axis=0
)

knockout_mean_abs = np.mean(
    np.abs(
        raw_oof_shap_values[
            knockout_mask, :
        ]
    ),
    axis=0
)


# ------------------------------------------------------------
# 2. Phase difference
# Knockout - Group
# ------------------------------------------------------------

phase_difference = (
    knockout_mean_abs
    - group_mean_abs
)


# ------------------------------------------------------------
# 3. Build table
# ------------------------------------------------------------

raw_shap_importance_df = pd.DataFrame({
    "Feature": feature_names_raw,
    "Overall_Mean_Abs_SHAP":
        overall_mean_abs,
    "Group_Mean_Abs_SHAP":
        group_mean_abs,
    "Knockout_Mean_Abs_SHAP":
        knockout_mean_abs,
    "Phase_Difference_KO_minus_Group":
        phase_difference
})


# Rank according to overall importance
raw_shap_importance_df = (
    raw_shap_importance_df
    .sort_values(
        "Overall_Mean_Abs_SHAP",
        ascending=False
    )
    .reset_index(drop=True)
)

raw_shap_importance_df.insert(
    0,
    "Rank",
    np.arange(
        1,
        len(raw_shap_importance_df) + 1
    )
)


# ------------------------------------------------------------
# 4. Display
# ------------------------------------------------------------

print("\nSAMPLE SIZES")
print("=" * 85)

print(
    "Overall:",
    len(df_raw)
)

print(
    "Group Stage:",
    group_mask.sum()
)

print(
    "Knockout Stage:",
    knockout_mask.sum()
)


print("\nOOF SHAP IMPORTANCE")
print("=" * 85)

display(
    raw_shap_importance_df.round(4)
)


# ------------------------------------------------------------
# 5. Audit weighted overall relationship
# ------------------------------------------------------------

weighted_check = (
    (
        group_mean_abs * group_mask.sum()
    )
    +
    (
        knockout_mean_abs
        * knockout_mask.sum()
    )
) / len(df_raw)

max_weighted_error = np.max(
    np.abs(
        weighted_check
        - overall_mean_abs
    )
)

print("\nAUDIT")
print("=" * 85)

print(
    "Maximum weighted overall discrepancy:",
    max_weighted_error
)

print(
    "Weighted phase means reproduce overall:",
    bool(
        np.allclose(
            weighted_check,
            overall_mean_abs
        )
    )
)


# ------------------------------------------------------------
# 6. Save preliminary importance table
# ------------------------------------------------------------

RAW_SHAP_IMPORTANCE_PATH = (
    RESULTS_DIR
    / "Sensitivity_Raw_XGBoost_OOF_SHAP_Importance_PreBootstrap.xlsx"
)

raw_shap_importance_df.to_excel(
    RAW_SHAP_IMPORTANCE_PATH,
    index=False
)

print("\nSaved:")
print(RAW_SHAP_IMPORTANCE_PATH)


In [ ]:
# ============================================================
# STEP 12: 5,000 MATCH-LEVEL BOOTSTRAP FOR OOF SHAP
# RAW-TOTAL XGBOOST SENSITIVITY
# ============================================================

import numpy as np
import pandas as pd

N_SHAP_BOOT = 5000
SHAP_BOOT_SEED = 42

rng_shap = np.random.default_rng(
    SHAP_BOOT_SEED
)

feature_names_raw = list(X_raw.columns)

match_array = np.asarray(groups_raw)
phase_array = (
    df_raw["Tournament_Phase"]
    .astype(str)
    .to_numpy()
)

unique_matches = np.unique(
    match_array
)

n_features = len(
    feature_names_raw
)

print(
    "RAW-TOTAL XGBOOST — "
    "MATCH-LEVEL SHAP BOOTSTRAP"
)
print("=" * 85)

print(
    "Bootstrap repetitions:",
    N_SHAP_BOOT
)

print(
    "Unique matches:",
    len(unique_matches)
)

print(
    "Features:",
    n_features
)


# ------------------------------------------------------------
# Storage
# ------------------------------------------------------------

boot_overall_shap = np.full(
    (N_SHAP_BOOT, n_features),
    np.nan
)

boot_group_shap = np.full(
    (N_SHAP_BOOT, n_features),
    np.nan
)

boot_knockout_shap = np.full(
    (N_SHAP_BOOT, n_features),
    np.nan
)

boot_phase_difference = np.full(
    (N_SHAP_BOOT, n_features),
    np.nan
)


# ------------------------------------------------------------
# Bootstrap matches
# ------------------------------------------------------------

for b in range(N_SHAP_BOOT):

    sampled_matches = rng_shap.choice(
        unique_matches,
        size=len(unique_matches),
        replace=True
    )

    sampled_indices = np.concatenate([
        np.where(
            match_array == match_id
        )[0]
        for match_id in sampled_matches
    ])

    shap_boot = raw_oof_shap_values[
        sampled_indices, :
    ]

    phase_boot = phase_array[
        sampled_indices
    ]

    abs_shap_boot = np.abs(
        shap_boot
    )

    group_boot_mask = (
        phase_boot == "Group Stage"
    )

    knockout_boot_mask = (
        phase_boot == "Knockout Stage"
    )

    # Skip only if a bootstrap sample
    # somehow contains no observations
    # from one phase
    if (
        group_boot_mask.sum() == 0
        or knockout_boot_mask.sum() == 0
    ):
        continue

    overall_boot = np.mean(
        abs_shap_boot,
        axis=0
    )

    group_boot = np.mean(
        abs_shap_boot[
            group_boot_mask, :
        ],
        axis=0
    )

    knockout_boot = np.mean(
        abs_shap_boot[
            knockout_boot_mask, :
        ],
        axis=0
    )

    boot_overall_shap[
        b, :
    ] = overall_boot

    boot_group_shap[
        b, :
    ] = group_boot

    boot_knockout_shap[
        b, :
    ] = knockout_boot

    boot_phase_difference[
        b, :
    ] = (
        knockout_boot
        - group_boot
    )


# ------------------------------------------------------------
# Summarize
# ------------------------------------------------------------

raw_shap_bootstrap_rows = []

for j, feature in enumerate(
    feature_names_raw
):

    overall_values = (
        boot_overall_shap[:, j]
    )

    group_values = (
        boot_group_shap[:, j]
    )

    knockout_values = (
        boot_knockout_shap[:, j]
    )

    difference_values = (
        boot_phase_difference[:, j]
    )

    valid = np.isfinite(
        difference_values
    )

    overall_values = (
        overall_values[valid]
    )

    group_values = (
        group_values[valid]
    )

    knockout_values = (
        knockout_values[valid]
    )

    difference_values = (
        difference_values[valid]
    )

    point_row = (
        raw_shap_importance_df
        .loc[
            raw_shap_importance_df[
                "Feature"
            ] == feature
        ]
        .iloc[0]
    )

    diff_lower = np.percentile(
        difference_values,
        2.5
    )

    diff_upper = np.percentile(
        difference_values,
        97.5
    )

    raw_shap_bootstrap_rows.append({

        "Feature": feature,

        "Overall_Mean_Abs_SHAP":
            point_row[
                "Overall_Mean_Abs_SHAP"
            ],

        "Overall_CI_Lower":
            np.percentile(
                overall_values,
                2.5
            ),

        "Overall_CI_Upper":
            np.percentile(
                overall_values,
                97.5
            ),

        "Group_Mean_Abs_SHAP":
            point_row[
                "Group_Mean_Abs_SHAP"
            ],

        "Group_CI_Lower":
            np.percentile(
                group_values,
                2.5
            ),

        "Group_CI_Upper":
            np.percentile(
                group_values,
                97.5
            ),

        "Knockout_Mean_Abs_SHAP":
            point_row[
                "Knockout_Mean_Abs_SHAP"
            ],

        "Knockout_CI_Lower":
            np.percentile(
                knockout_values,
                2.5
            ),

        "Knockout_CI_Upper":
            np.percentile(
                knockout_values,
                97.5
            ),

        "Phase_Difference_KO_minus_Group":
            point_row[
                "Phase_Difference_KO_minus_Group"
            ],

        "Phase_Difference_CI_Lower":
            diff_lower,

        "Phase_Difference_CI_Upper":
            diff_upper,

        "Phase_Difference_CI_Excludes_Zero":
            bool(
                (diff_lower > 0)
                or
                (diff_upper < 0)
            ),

        "Valid_Bootstraps":
            len(
                difference_values
            )
    })


raw_shap_bootstrap_df = pd.DataFrame(
    raw_shap_bootstrap_rows
)


# ------------------------------------------------------------
# Sort by overall importance
# ------------------------------------------------------------

raw_shap_bootstrap_df = (
    raw_shap_bootstrap_df
    .sort_values(
        "Overall_Mean_Abs_SHAP",
        ascending=False
    )
    .reset_index(drop=True)
)

raw_shap_bootstrap_df.insert(
    0,
    "Rank",
    np.arange(
        1,
        len(
            raw_shap_bootstrap_df
        ) + 1
    )
)


# ------------------------------------------------------------
# Display compact table
# ------------------------------------------------------------

display_columns = [
    "Rank",
    "Feature",
    "Overall_Mean_Abs_SHAP",
    "Overall_CI_Lower",
    "Overall_CI_Upper",
    "Phase_Difference_KO_minus_Group",
    "Phase_Difference_CI_Lower",
    "Phase_Difference_CI_Upper",
    "Phase_Difference_CI_Excludes_Zero",
    "Valid_Bootstraps"
]

print(
    "\nSHAP BOOTSTRAP RESULTS"
)
print("=" * 85)

display(
    raw_shap_bootstrap_df[
        display_columns
    ].round(4)
)


# ------------------------------------------------------------
# Important audit
# ------------------------------------------------------------

print("\nBOOTSTRAP AUDIT")
print("=" * 85)

print(
    "Minimum valid bootstrap samples:",
    raw_shap_bootstrap_df[
        "Valid_Bootstraps"
    ].min()
)

print(
    "\nFeatures whose phase-difference "
    "95% CI excludes zero:"
)

phase_supported = (
    raw_shap_bootstrap_df[
        raw_shap_bootstrap_df[
            "Phase_Difference_CI_Excludes_Zero"
        ]
    ][
        [
            "Feature",
            "Phase_Difference_KO_minus_Group",
            "Phase_Difference_CI_Lower",
            "Phase_Difference_CI_Upper"
        ]
    ]
)

display(
    phase_supported.round(4)
)


# ------------------------------------------------------------
# Save final sensitivity SHAP table
# ------------------------------------------------------------

RAW_SHAP_FINAL_PATH = (
    RESULTS_DIR
    / "Sensitivity_Raw_XGBoost_OOF_SHAP_Importance.xlsx"
)

raw_shap_bootstrap_df.to_excel(
    RAW_SHAP_FINAL_PATH,
    index=False
)

print("\nSaved:")
print(RAW_SHAP_FINAL_PATH)


## Primary versus sensitivity comparison

The following analyses compare the per-90 primary results with the raw-total sensitivity results.

The purpose is to assess robustness to the representation of exposure-dependent performance indicators.


In [ ]:
# ============================================================
# STEP 13A: CHECK AVAILABILITY OF PRIMARY PER-90 RESULTS
# ============================================================

print("PRIMARY vs SENSITIVITY COMPARISON — OBJECT CHECK")
print("=" * 80)

objects_to_check = [
    "oof_predictions",
    "oof_shap_values",
    "shap_importance_df",
    "shap_bootstrap_df"
]

for obj in objects_to_check:
    print(
        f"{obj}:",
        "AVAILABLE"
        if obj in globals()
        else "NOT AVAILABLE"
    )


print("\nRAW SENSITIVITY OBJECTS")
print("=" * 80)

raw_objects = [
    "raw_oof_predictions",
    "raw_oof_shap_values",
    "raw_shap_importance_df",
    "raw_shap_bootstrap_df"
]

for obj in raw_objects:
    print(
        f"{obj}:",
        "AVAILABLE"
        if obj in globals()
        else "NOT AVAILABLE"
    )


In [ ]:
# ============================================================
# STEP 13B: LOCATE SAVED PRIMARY PER-90 RESULT FILES
# ============================================================

from pathlib import Path

print("SAVED PRIMARY RESULT FILES")
print("=" * 85)

folders_to_check = [
    RESULTS_DIR,
    RESULTS_DIR
]

for folder in folders_to_check:

    print(f"\nFOLDER: {folder}")
    print("-" * 85)

    if not folder.exists():
        print("Folder does not exist.")
        continue

    files = sorted([
        f for f in folder.iterdir()
        if f.is_file()
    ])

    for f in files:
        print(f.name)

print("\nSTEP 13B COMPLETE")


In [ ]:
# ============================================================
# STEP 13C: LOAD SAVED PRIMARY PER-90 RESULTS
# + VERIFY ROW-LEVEL SHAP ALIGNMENT
# ============================================================

import pandas as pd
import numpy as np

TABLE_DIR = RESULTS_DIR
SHAP_DIR = RESULTS_DIR


# ------------------------------------------------------------
# 1. Load primary model-performance results
# ------------------------------------------------------------

primary_performance = pd.read_excel(
    TABLE_DIR / "Primary_Model_Performance.xlsx"
)

primary_bootstrap_ci = pd.read_excel(
    TABLE_DIR / "Primary_Model_Performance_Bootstrap_CI.xlsx"
)

primary_paired_models = pd.read_excel(
    TABLE_DIR / "Paired_Model_Comparisons.xlsx"
)

primary_phase_performance = pd.read_excel(
    TABLE_DIR / "Phase_Specific_Model_Performance.xlsx"
)

primary_shap_importance = pd.read_excel(
    TABLE_DIR / "Primary_XGBoost_OOF_SHAP_Importance.xlsx"
)


# ------------------------------------------------------------
# 2. Load primary row-level SHAP
# ------------------------------------------------------------

primary_shap_matrix = pd.read_csv(
    SHAP_DIR / "Primary_XGBoost_OOF_SHAP_Matrix.csv"
)

primary_shap_feature_values = pd.read_csv(
    SHAP_DIR / "Primary_XGBoost_OOF_SHAP_Feature_Values.csv"
)

primary_shap_row_level = pd.read_excel(
    SHAP_DIR / "Primary_XGBoost_OOF_SHAP_Row_Level.xlsx"
)


# ------------------------------------------------------------
# 3. Basic shape audit
# ------------------------------------------------------------

print("PRIMARY RESULT LOADING AUDIT")
print("=" * 85)

print(
    "Primary performance:",
    primary_performance.shape
)

print(
    "Primary bootstrap CI:",
    primary_bootstrap_ci.shape
)

print(
    "Primary paired models:",
    primary_paired_models.shape
)

print(
    "Primary phase performance:",
    primary_phase_performance.shape
)

print(
    "Primary SHAP importance:",
    primary_shap_importance.shape
)

print(
    "Primary SHAP matrix:",
    primary_shap_matrix.shape
)

print(
    "Primary SHAP feature values:",
    primary_shap_feature_values.shape
)

print(
    "Primary row-level SHAP:",
    primary_shap_row_level.shape
)


# ------------------------------------------------------------
# 4. Check primary SHAP matrix against raw sensitivity
# ------------------------------------------------------------

print("\nSHAP ROW ALIGNMENT CHECK")
print("=" * 85)

print(
    "Primary SHAP rows:",
    len(primary_shap_matrix)
)

print(
    "Raw sensitivity SHAP rows:",
    raw_oof_shap_values.shape[0]
)

print(
    "Same number of observations:",
    len(primary_shap_matrix)
    == raw_oof_shap_values.shape[0]
)


# ------------------------------------------------------------
# 5. Inspect row-level metadata columns
# ------------------------------------------------------------

print("\nPRIMARY ROW-LEVEL SHAP COLUMNS")
print("=" * 85)

print(
    primary_shap_row_level.columns.tolist()
)


# ------------------------------------------------------------
# 6. Display primary performance
# ------------------------------------------------------------

print("\nPRIMARY PER-90 PERFORMANCE")
print("=" * 85)

display(
    primary_performance.round(4)
)


print("\nRAW-TOTAL SENSITIVITY PERFORMANCE")
print("=" * 85)

display(
    raw_performance_df.round(4)
)


print("\nSTEP 13C COMPLETE")


In [ ]:
# ============================================================
# STEP 13D: PAIRED PRIMARY PER-90 vs RAW-TOTAL XGBOOST
# 5,000 MATCH-LEVEL BOOTSTRAP
# ============================================================

import numpy as np
import pandas as pd

print("PRIMARY PER-90 vs RAW-TOTAL XGBOOST")
print("=" * 90)


# ------------------------------------------------------------
# 1. Verify exact row alignment using metadata
# ------------------------------------------------------------

primary_keys = (
    primary_shap_row_level[
        ["Match_ID", "Team", "Opponent"]
    ]
    .astype(str)
    .reset_index(drop=True)
)

raw_keys = (
    df_raw[
        ["Match_ID", "Team", "Opponent"]
    ]
    .astype(str)
    .reset_index(drop=True)
)

alignment = (
    primary_keys == raw_keys
).all(axis=1)

print(
    "Exactly aligned rows:",
    int(alignment.sum()),
    "/",
    len(alignment)
)

print(
    "All rows aligned:",
    bool(alignment.all())
)


# ------------------------------------------------------------
# 2. Verify outcome alignment
# ------------------------------------------------------------

primary_y = (
    primary_shap_row_level[
        "Win_Binary"
    ]
    .astype(int)
    .to_numpy()
)

raw_y = np.asarray(
    y_raw,
    dtype=int
)

print(
    "Outcome identical:",
    bool(
        np.array_equal(
            primary_y,
            raw_y
        )
    )
)


# ------------------------------------------------------------
# STOP automatically if alignment is wrong
# ------------------------------------------------------------

assert alignment.all(), (
    "STOP: Primary and raw rows are not aligned."
)

assert np.array_equal(
    primary_y,
    raw_y
), "STOP: Outcomes are not aligned."


# ------------------------------------------------------------
# 3. Retrieve XGBoost OOF probabilities
# ------------------------------------------------------------

per90_prob = (
    primary_shap_row_level[
        "OOF_XGBoost_Win_Probability"
    ]
    .astype(float)
    .to_numpy()
)

raw_prob = np.asarray(
    raw_oof_predictions["XGBoost"],
    dtype=float
)


print(
    "\nPrimary probability missing:",
    np.isnan(per90_prob).sum()
)

print(
    "Raw probability missing:",
    np.isnan(raw_prob).sum()
)


# ------------------------------------------------------------
# 4. Calculate point-estimate metrics
# ------------------------------------------------------------

per90_metrics = bootstrap_metrics(
    raw_y,
    per90_prob
)

raw_metrics = bootstrap_metrics(
    raw_y,
    raw_prob
)


comparison_metrics = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "ROC_AUC",
    "Brier",
    "ECE"
]

point_rows = []

for metric in comparison_metrics:

    point_rows.append({
        "Metric": metric,
        "Per90": per90_metrics[metric],
        "Raw_Total": raw_metrics[metric],
        "Difference_Per90_minus_Raw":
            per90_metrics[metric]
            - raw_metrics[metric]
    })

xgb_representation_point_df = pd.DataFrame(
    point_rows
)

print("\nPOINT ESTIMATES")
print("=" * 90)

display(
    xgb_representation_point_df.round(4)
)


# ------------------------------------------------------------
# 5. Paired match-level bootstrap
# ------------------------------------------------------------

N_REP_BOOT = 5000
REP_BOOT_SEED = 42

rng_rep = np.random.default_rng(
    REP_BOOT_SEED
)

match_ids = np.asarray(
    groups_raw
)

unique_match_ids = np.unique(
    match_ids
)

boot_differences = {
    metric: []
    for metric in comparison_metrics
}


for b in range(N_REP_BOOT):

    sampled_matches = rng_rep.choice(
        unique_match_ids,
        size=len(unique_match_ids),
        replace=True
    )

    sampled_indices = np.concatenate([
        np.where(
            match_ids == match_id
        )[0]
        for match_id in sampled_matches
    ])

    y_boot = raw_y[
        sampled_indices
    ]

    if len(np.unique(y_boot)) < 2:
        continue

    per90_boot = bootstrap_metrics(
        y_boot,
        per90_prob[sampled_indices]
    )

    raw_boot = bootstrap_metrics(
        y_boot,
        raw_prob[sampled_indices]
    )

    for metric in comparison_metrics:

        boot_differences[
            metric
        ].append(
            per90_boot[metric]
            - raw_boot[metric]
        )


# ------------------------------------------------------------
# 6. Summarize paired CIs
# ------------------------------------------------------------

paired_representation_rows = []

for metric in comparison_metrics:

    values = np.asarray(
        boot_differences[metric]
    )

    point_difference = (
        per90_metrics[metric]
        - raw_metrics[metric]
    )

    ci_lower = np.percentile(
        values,
        2.5
    )

    ci_upper = np.percentile(
        values,
        97.5
    )

    paired_representation_rows.append({
        "Metric": metric,
        "Per90":
            per90_metrics[metric],
        "Raw_Total":
            raw_metrics[metric],
        "Difference_Per90_minus_Raw":
            point_difference,
        "CI_Lower":
            ci_lower,
        "CI_Upper":
            ci_upper,
        "CI_Excludes_Zero":
            bool(
                (ci_lower > 0)
                or
                (ci_upper < 0)
            ),
        "Valid_Bootstraps":
            len(values)
    })


xgb_representation_bootstrap_df = pd.DataFrame(
    paired_representation_rows
)


print(
    "\nPAIRED REPRESENTATION COMPARISON"
)
print("=" * 90)

display(
    xgb_representation_bootstrap_df.round(4)
)


print("\nINTERPRETATION OF DIFFERENCE")
print("=" * 90)

print(
    "Difference = per-90 XGBoost - raw-total XGBoost"
)

print(
    "For Accuracy, Precision, Recall, F1 and ROC-AUC:"
    " positive = higher under per-90."
)

print(
    "For Brier and ECE:"
    " negative = lower/better under per-90."
)


# ------------------------------------------------------------
# 7. Save
# ------------------------------------------------------------

REPRESENTATION_PATH = (
    RESULTS_DIR
    / "Primary_Per90_vs_Raw_XGBoost_Paired_Comparison.xlsx"
)

xgb_representation_bootstrap_df.to_excel(
    REPRESENTATION_PATH,
    index=False
)

print("\nSaved:")
print(REPRESENTATION_PATH)


In [ ]:
# ============================================================
# STEP 14: PRIMARY PER-90 vs RAW-TOTAL SHAP ROBUSTNESS
# Feature-importance ranking comparison
# ============================================================

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

print("PRIMARY PER-90 vs RAW-TOTAL XGBOOST SHAP ROBUSTNESS")
print("=" * 95)


# ------------------------------------------------------------
# 1. Feature-name mapping
# ------------------------------------------------------------

feature_map = {
    "Ranking_Difference":
        "Ranking_Difference",

    "Attempts_on_Target_per90":
        "Attempts_on_Target",

    "Corner_Kicks_per90":
        "Corner_Kicks",

    "Crosses_per90":
        "Crosses",

    "Ball_Possession_%":
        "Ball_Possession_Percent",

    "Completed_Passes_per90":
        "Completed_Passes",

    "Completed_Line_Breaks_per90":
        "Completed_Line_Breaks",

    "Defensive_Pressures_per90":
        "Defensive_Pressures",

    "Forced_Turnovers_per90":
        "Forced_Turnovers",

    "Second_Balls_per90":
        "Second_Balls",

    "Saves_per90":
        "Saves",

    "Save_percentage":
        "Save_Percentage",

    "Distance_per_90_recalculated":
        "Distance_Covered_km",

    "Zone4_per90":
        "Zone4_Low_Speed_Sprinting"
}


# ------------------------------------------------------------
# 2. Calculate mean absolute SHAP directly from matrices
# ------------------------------------------------------------

primary_shap_matrix_np = (
    primary_shap_matrix
    .to_numpy(dtype=float)
)

raw_shap_matrix_np = np.asarray(
    raw_oof_shap_values,
    dtype=float
)


primary_feature_names = list(
    primary_shap_matrix.columns
)

raw_feature_names = list(
    X_raw.columns
)


# ------------------------------------------------------------
# 3. Build comparison table
# ------------------------------------------------------------

comparison_rows = []

for primary_feature, raw_feature in feature_map.items():

    primary_j = primary_feature_names.index(
        primary_feature
    )

    raw_j = raw_feature_names.index(
        raw_feature
    )

    primary_mean = np.mean(
        np.abs(
            primary_shap_matrix_np[:, primary_j]
        )
    )

    raw_mean = np.mean(
        np.abs(
            raw_shap_matrix_np[:, raw_j]
        )
    )

    comparison_rows.append({
        "Conceptual_Feature": raw_feature,
        "Primary_Per90_Feature": primary_feature,
        "Raw_Total_Feature": raw_feature,
        "Per90_Mean_Abs_SHAP": primary_mean,
        "Raw_Mean_Abs_SHAP": raw_mean
    })


shap_representation_comparison = pd.DataFrame(
    comparison_rows
)


# ------------------------------------------------------------
# 4. Calculate ranks independently
# ------------------------------------------------------------

shap_representation_comparison[
    "Per90_Rank"
] = (
    shap_representation_comparison[
        "Per90_Mean_Abs_SHAP"
    ]
    .rank(
        method="min",
        ascending=False
    )
    .astype(int)
)

shap_representation_comparison[
    "Raw_Rank"
] = (
    shap_representation_comparison[
        "Raw_Mean_Abs_SHAP"
    ]
    .rank(
        method="min",
        ascending=False
    )
    .astype(int)
)

shap_representation_comparison[
    "Rank_Change_Raw_minus_Per90"
] = (
    shap_representation_comparison[
        "Raw_Rank"
    ]
    -
    shap_representation_comparison[
        "Per90_Rank"
    ]
)


# ------------------------------------------------------------
# 5. Spearman rank correlation
# ------------------------------------------------------------

rho, rho_p = spearmanr(
    shap_representation_comparison[
        "Per90_Rank"
    ],
    shap_representation_comparison[
        "Raw_Rank"
    ]
)


# ------------------------------------------------------------
# 6. Top-feature overlap
# ------------------------------------------------------------

def get_top_features(df, rank_col, n):

    return set(
        df.loc[
            df[rank_col] <= n,
            "Conceptual_Feature"
        ]
    )


top3_per90 = get_top_features(
    shap_representation_comparison,
    "Per90_Rank",
    3
)

top3_raw = get_top_features(
    shap_representation_comparison,
    "Raw_Rank",
    3
)

top5_per90 = get_top_features(
    shap_representation_comparison,
    "Per90_Rank",
    5
)

top5_raw = get_top_features(
    shap_representation_comparison,
    "Raw_Rank",
    5
)


top3_overlap = len(
    top3_per90.intersection(top3_raw)
)

top5_overlap = len(
    top5_per90.intersection(top5_raw)
)


# ------------------------------------------------------------
# 7. Sort for display
# ------------------------------------------------------------

shap_representation_comparison = (
    shap_representation_comparison
    .sort_values(
        "Per90_Rank"
    )
    .reset_index(drop=True)
)


print("\nFEATURE IMPORTANCE COMPARISON")
print("=" * 95)

display(
    shap_representation_comparison[
        [
            "Conceptual_Feature",
            "Per90_Mean_Abs_SHAP",
            "Per90_Rank",
            "Raw_Mean_Abs_SHAP",
            "Raw_Rank",
            "Rank_Change_Raw_minus_Per90"
        ]
    ].round(4)
)


print("\nRANKING ROBUSTNESS")
print("=" * 95)

print(
    "Spearman rank correlation:",
    round(rho, 4)
)

print(
    "Spearman p-value:",
    f"{rho_p:.6g}"
)

print(
    "Top-3 overlap:",
    f"{top3_overlap}/3"
)

print(
    "Top-5 overlap:",
    f"{top5_overlap}/5"
)

print(
    "\nPer-90 top 3:",
    sorted(top3_per90)
)

print(
    "Raw-total top 3:",
    sorted(top3_raw)
)

print(
    "\nPer-90 top 5:",
    sorted(top5_per90)
)

print(
    "Raw-total top 5:",
    sorted(top5_raw)
)


# ------------------------------------------------------------
# 8. Save
# ------------------------------------------------------------

SHAP_REPRESENTATION_PATH = (
    RESULTS_DIR
    / "Primary_Per90_vs_Raw_SHAP_Robustness.xlsx"
)

shap_representation_comparison.to_excel(
    SHAP_REPRESENTATION_PATH,
    index=False
)

print("\nSaved:")
print(SHAP_REPRESENTATION_PATH)


## Save sensitivity-analysis outputs

The final model-performance, SHAP, robustness, and row-level reproducibility outputs are written to the repository `results/` directory.


In [ ]:
# ============================================================
# STEP 15: SAVE COMPLETE RAW-TOTAL SENSITIVITY OUTPUTS
# ============================================================

import pandas as pd
import numpy as np

TABLE_DIR = RESULTS_DIR
SHAP_DIR = RESULTS_DIR

TABLE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

SHAP_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# 1. Save pooled model performance
# ------------------------------------------------------------

raw_performance_path = (
    TABLE_DIR
    / "Sensitivity_Raw_Model_Performance.xlsx"
)

raw_performance_df.to_excel(
    raw_performance_path,
    index=False
)


# ------------------------------------------------------------
# 2. Save phase-specific performance
# ------------------------------------------------------------

raw_phase_path = (
    TABLE_DIR
    / "Sensitivity_Raw_Phase_Specific_Model_Performance.xlsx"
)

raw_phase_performance_df.to_excel(
    raw_phase_path,
    index=False
)


# ------------------------------------------------------------
# 3. Save outer-fold results
# ------------------------------------------------------------

raw_fold_path = (
    TABLE_DIR
    / "Sensitivity_Raw_Outer_Fold_Results.xlsx"
)

raw_fold_results_df.to_excel(
    raw_fold_path,
    index=False
)


# ------------------------------------------------------------
# 4. Save best hyperparameters
# ------------------------------------------------------------

raw_params_path = (
    TABLE_DIR
    / "Sensitivity_Raw_Best_Parameters.xlsx"
)

raw_best_parameters_df.to_excel(
    raw_params_path,
    index=False
)


# ------------------------------------------------------------
# 5. Create row-level raw SHAP output
# ------------------------------------------------------------

raw_shap_row_level = pd.DataFrame({
    "Match_ID":
        df_raw["Match_ID"].values,

    "Team":
        df_raw["Team"].values,

    "Opponent":
        df_raw["Opponent"].values,

    "Tournament_Phase":
        df_raw["Tournament_Phase"].values,

    "Win_Binary":
        raw_y,

    "OOF_XGBoost_Win_Probability":
        raw_prob
})


# Add raw feature values
for j, feature in enumerate(
    feature_names_raw
):

    raw_shap_row_level[
        f"VALUE__{feature}"
    ] = raw_oof_shap_feature_values[
        :, j
    ]


# Add SHAP values
for j, feature in enumerate(
    feature_names_raw
):

    raw_shap_row_level[
        f"SHAP__{feature}"
    ] = raw_oof_shap_values[
        :, j
    ]


raw_row_level_path = (
    SHAP_DIR
    / "Sensitivity_Raw_XGBoost_OOF_SHAP_Row_Level.xlsx"
)

raw_shap_row_level.to_excel(
    raw_row_level_path,
    index=False
)


# ------------------------------------------------------------
# 6. Save raw SHAP matrix
# ------------------------------------------------------------

raw_shap_matrix_df = pd.DataFrame(
    raw_oof_shap_values,
    columns=feature_names_raw
)

raw_shap_matrix_path = (
    SHAP_DIR
    / "Sensitivity_Raw_XGBoost_OOF_SHAP_Matrix.csv"
)

raw_shap_matrix_df.to_csv(
    raw_shap_matrix_path,
    index=False
)


# ------------------------------------------------------------
# 7. Save raw SHAP feature-value matrix
# ------------------------------------------------------------

raw_shap_feature_values_df = pd.DataFrame(
    raw_oof_shap_feature_values,
    columns=feature_names_raw
)

raw_feature_values_path = (
    SHAP_DIR
    / "Sensitivity_Raw_XGBoost_OOF_SHAP_Feature_Values.csv"
)

raw_shap_feature_values_df.to_csv(
    raw_feature_values_path,
    index=False
)


# ------------------------------------------------------------
# 8. Final audit
# ------------------------------------------------------------

print("RAW-TOTAL SENSITIVITY — FINAL OUTPUT AUDIT")
print("=" * 90)

files_saved = [
    raw_performance_path,
    raw_phase_path,
    raw_fold_path,
    raw_params_path,
    raw_row_level_path,
    raw_shap_matrix_path,
    raw_feature_values_path
]

for path in files_saved:

    print(
        path.name,
        "->",
        "SAVED"
        if path.exists()
        else "MISSING"
    )


print("\nROW-LEVEL AUDIT")
print("=" * 90)

print(
    "Rows:",
    len(raw_shap_row_level)
)

print(
    "Columns:",
    len(raw_shap_row_level.columns)
)

print(
    "SHAP matrix:",
    raw_shap_matrix_df.shape
)

print(
    "Feature-value matrix:",
    raw_shap_feature_values_df.shape
)

print(
    "Missing SHAP values:",
    raw_shap_matrix_df.isna().sum().sum()
)

print(
    "Missing OOF probabilities:",
    raw_shap_row_level[
        "OOF_XGBoost_Win_Probability"
    ].isna().sum()
)

print("\nSTEP 15 COMPLETE")


## Final SHAP visualization

The final raw-total XGBoost SHAP beeswarm is saved to the repository `figures/` directory.


In [ ]:
# ============================================================
# STEP 16: FINAL RAW-TOTAL XGBOOST OOF SHAP BEESWARM
# ============================================================

import shap
import matplotlib.pyplot as plt
from pathlib import Path

FIGURE_DIR = FIGURES_DIR
FIGURE_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# 1. Reader-friendly feature labels
# ------------------------------------------------------------

raw_display_names = {
    "Ranking_Difference":
        "FIFA ranking difference",

    "Attempts_on_Target":
        "Attempts on target",

    "Save_Percentage":
        "Save percentage",

    "Ball_Possession_Percent":
        "Ball possession",

    "Crosses":
        "Crosses",

    "Defensive_Pressures":
        "Defensive pressures",

    "Second_Balls":
        "Second balls",

    "Distance_Covered_km":
        "Distance covered",

    "Saves":
        "Saves",

    "Zone4_Low_Speed_Sprinting":
        "Zone 4 low-speed sprinting",

    "Corner_Kicks":
        "Corner kicks",

    "Completed_Line_Breaks":
        "Completed line breaks",

    "Forced_Turnovers":
        "Forced turnovers",

    "Completed_Passes":
        "Completed passes"
}


# ------------------------------------------------------------
# 2. Order features by RAW overall mean |SHAP|
# ------------------------------------------------------------

ordered_raw_features = (
    raw_shap_bootstrap_df
    .sort_values(
        "Overall_Mean_Abs_SHAP",
        ascending=False
    )["Feature"]
    .tolist()
)

ordered_indices = [
    feature_names_raw.index(feature)
    for feature in ordered_raw_features
]

shap_plot_values = (
    raw_oof_shap_values[
        :, ordered_indices
    ]
)

feature_plot_values = (
    raw_oof_shap_feature_values[
        :, ordered_indices
    ]
)

plot_names = [
    raw_display_names.get(
        feature,
        feature
    )
    for feature in ordered_raw_features
]


# ------------------------------------------------------------
# 3. Construct SHAP Explanation
# ------------------------------------------------------------

raw_explanation = shap.Explanation(
    values=shap_plot_values,
    data=feature_plot_values,
    feature_names=plot_names
)


# ------------------------------------------------------------
# 4. Create beeswarm
# ------------------------------------------------------------

plt.figure(
    figsize=(10.5, 7.5)
)

shap.plots.beeswarm(
    raw_explanation,
    max_display=14,
    show=False
)

ax = plt.gca()

ax.set_xlabel(
    "SHAP value (← toward non-win | toward win →)",
    fontsize=11
)

ax.set_ylabel(
    ""
)

ax.set_title(
    "Raw-total sensitivity analysis: XGBoost OOF SHAP",
    fontsize=13,
    pad=14
)

ax.axvline(
    0,
    linewidth=0.8,
    linestyle="--"
)

plt.tight_layout()


# ------------------------------------------------------------
# 5. Save PNG + PDF
# ------------------------------------------------------------

RAW_BEESWARM_PNG = (
    FIGURE_DIR
    / "Sensitivity_Raw_XGBoost_OOF_SHAP_Beeswarm_FINAL.png"
)

RAW_BEESWARM_PDF = (
    FIGURE_DIR
    / "Sensitivity_Raw_XGBoost_OOF_SHAP_Beeswarm_FINAL.pdf"
)

plt.savefig(
    RAW_BEESWARM_PNG,
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    RAW_BEESWARM_PDF,
    bbox_inches="tight"
)

plt.show()


# ------------------------------------------------------------
# 6. Audit
# ------------------------------------------------------------

print(
    "RAW-TOTAL SHAP BEESWARM — FINAL AUDIT"
)
print("=" * 90)

print(
    "Features displayed:",
    len(plot_names)
)

print(
    "Top 3:",
    plot_names[:3]
)

print(
    "\nPNG:",
    RAW_BEESWARM_PNG,
    "->",
    "SAVED"
    if RAW_BEESWARM_PNG.exists()
    else "MISSING"
)

print(
    "\nPDF:",
    RAW_BEESWARM_PDF,
    "->",
    "SAVED"
    if RAW_BEESWARM_PDF.exists()
    else "MISSING"
)

print(
    "\nInterpretation:"
)

print(
    "Positive SHAP values push the model output toward WIN."
)

print(
    "Negative SHAP values push the model output toward NON-WIN."
)

print(
    "Blue/red represents low/high predictor values, NOT outcome."
)

print(
    "For FIFA ranking difference: positive values mean "
    "the focal team was better ranked than its opponent."
)

print("\nSTEP 16 COMPLETE")


## Software environment

The final cell records the software environment used for the analysis to support computational reproducibility.


In [ ]:
# ============================================================
# STEP 17: RECORD EXACT SOFTWARE ENVIRONMENT
# ============================================================

import sys
import platform
import importlib.metadata as metadata
import pandas as pd
from pathlib import Path

print("SOFTWARE ENVIRONMENT AUDIT")
print("=" * 80)

print("Python version:")
print(sys.version)

print("\nPlatform:")
print(platform.platform())

packages = [
    "numpy",
    "pandas",
    "scipy",
    "scikit-learn",
    "xgboost",
    "shap",
    "matplotlib",
    "openpyxl",
    "jupyter",
    "notebook",
    "ipykernel"
]

environment_rows = []

print("\nPACKAGE VERSIONS")
print("=" * 80)

for package in packages:
    try:
        version = metadata.version(package)
    except metadata.PackageNotFoundError:
        version = "Not installed / not detected"

    environment_rows.append({
        "Package": package,
        "Version": version
    })

    print(f"{package}: {version}")

environment_df = pd.DataFrame(environment_rows)


# ------------------------------------------------------------
# Save environment record
# ------------------------------------------------------------

REPRO_DIR = REPO_ROOT
REPRO_DIR.mkdir(
    parents=True,
    exist_ok=True
)

ENV_PATH = (
    REPRO_DIR
    / "Software_Environment.xlsx"
)

environment_df.to_excel(
    ENV_PATH,
    index=False
)


# Also save plain-text version
ENV_TXT_PATH = (
    REPRO_DIR
    / "Software_Environment.txt"
)

with open(
    ENV_TXT_PATH,
    "w",
    encoding="utf-8"
) as f:

    f.write("FIFA WORLD CUP 2026 EXPLAINABLE ML\n")
    f.write("SOFTWARE ENVIRONMENT\n")
    f.write("=" * 60 + "\n\n")

    f.write(f"Python: {sys.version}\n\n")
    f.write(f"Platform: {platform.platform()}\n\n")

    f.write("Packages:\n")

    for row in environment_rows:
        f.write(
            f"{row['Package']}=={row['Version']}\n"
        )


print("\nSAVED")
print("=" * 80)

print(ENV_PATH)
print(ENV_TXT_PATH)

print("\nSTEP 17 COMPLETE")
